# LambdaMART 排序学习选股策略

## 论文参考

- **标题**: Portfolio Models Based on Fundamental Analysis Using Learning to Rank
- **作者**: Lei Zhang
- **年份**: 2021

### 摘要

本文将排序学习 (Learning to Rank, LTR) 方法引入股票选股，
使用 LambdaMART 算法对股票进行排名而非直接预测收益率。
相比传统回归方法，排序学习更关注股票间的相对优劣而非绝对收益预测，
在选股场景下通常表现更优。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### LambdaMART 排序学习

**1. 问题定义**

给定日期 $t$ 的截面因子 $\{x_1, x_2, \ldots, x_N\}$，目标是学习排序函数 $f$ 使得:

$$f(x_i) > f(x_j) \iff r_{i,t+1} > r_{j,t+1}$$

即排序函数应与未来收益率排序一致。

**2. LambdaMART 损失函数**

基于 pairwise 思想，对每对 $(i, j)$ 计算 Lambda 梯度:

$$\lambda_{ij} = \frac{-\sigma}{1 + e^{\sigma(s_i - s_j)}} \cdot |\Delta \text{NDCG}_{ij}|$$

其中 $s_i = f(x_i)$ 是预测分数，$\Delta\text{NDCG}_{ij}$ 是交换 $i, j$ 位置后 NDCG 的变化量。

**3. 训练标签构造**

将截面内股票按未来收益率排名，分为 5 档 (0-4):
- 4: Top 20% (最高收益)
- 0: Bottom 20% (最低收益)

**4. LightGBM 实现**

使用 LightGBM 的 `objective='lambdarank'`，以 NDCG 为评估指标。

In [ ]:
# ============================================================
# Cell 3: 动画展示 - Pairwise 排序损失引导树分裂
# ============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

# 模拟 LambdaMART 训练过程
n_rounds = 20
n_stocks = 20

# 真实排名 (按未来收益)
true_ranks = np.arange(n_stocks)
np.random.shuffle(true_ranks)

# 模拟每轮预测排名逐步接近真实排名
frames = []
ndcg_scores = []

for r in range(n_rounds + 1):
    # 模型预测排名随训练轮次改善
    noise_level = 1.0 * (1 - r / n_rounds) ** 1.5
    pred_scores = true_ranks.astype(float) + np.random.randn(n_stocks) * noise_level * n_stocks * 0.3
    pred_ranks = np.argsort(np.argsort(-pred_scores))  # 降序排名

    # 计算 NDCG@5
    true_relevance = (n_stocks - true_ranks) / n_stocks  # 高排名高相关性
    top5_idx = np.argsort(-pred_scores)[:5]
    dcg = sum(true_relevance[top5_idx[i]] / np.log2(i + 2) for i in range(5))
    ideal_top5 = np.argsort(-true_relevance)[:5]
    idcg = sum(true_relevance[ideal_top5[i]] / np.log2(i + 2) for i in range(5))
    ndcg = dcg / idcg if idcg > 0 else 0
    ndcg_scores.append(ndcg)

    frames.append(go.Frame(
        data=[
            go.Bar(x=[f'Stock {i}' for i in range(n_stocks)],
                   y=pred_scores, name='预测得分',
                   marker_color=['#F44336' if i in top5_idx else '#2196F3' for i in range(n_stocks)]),
            go.Scatter(x=list(range(r+1)), y=ndcg_scores[:r+1],
                       mode='lines+markers', name='NDCG@5',
                       line=dict(color='#4CAF50', width=3), yaxis='y2'),
        ],
        name=f'Round {r}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='LambdaMART: Pairwise排序学习过程 (Top-5标红)'),
        yaxis=dict(title='预测得分'),
        yaxis2=dict(title='NDCG@5', overlaying='y', side='right', range=[0, 1.1]),
        height=500, width=1000,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 400}}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.backtest_engine import MultiStockBacktester
from shared.factors import (
    momentum, volatility, rsi, macd, bollinger_bands,
    price_to_ma, volume_price_corr, high_low_range, turnover_factor
)
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_factor_importance, plot_cumulative_comparison
)
from shared.constants import DEFAULT_START, DEFAULT_END, CSI300_CODE

print('所有模块导入成功')

In [ ]:
# ============================================================
# Cell 5: 数据获取 (20只股票)
# ============================================================
STOCK_POOL = [
    '600519', '601318', '600036', '000858', '601166',
    '600276', '601398', '600030', '000333', '002415',
    '600900', '601888', '000568', '002304', '601012',
    '600031', '603259', '600585', '000661', '601668',
]

try:
    stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
    print(f'成功获取 {len(stock_data)} 只股票数据')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    stock_data = {}
    for sym in STOCK_POOL:
        np.random.seed(hash(sym) % 2**31)
        price = 50 + np.cumsum(np.random.randn(len(dates)) * 0.5)
        price = np.maximum(price, 5)
        stock_data[sym] = pd.DataFrame({
            'open': price * (1 + np.random.randn(len(dates)) * 0.005),
            'close': price,
            'high': price * (1 + np.abs(np.random.randn(len(dates)) * 0.01)),
            'low': price * (1 - np.abs(np.random.randn(len(dates)) * 0.01)),
            'volume': np.random.randint(100000, 5000000, len(dates)).astype(float),
            'turnover': np.random.uniform(0.3, 5.0, len(dates)),
        }, index=dates)

# 基准
try:
    benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)
    benchmark_prices = benchmark['close']
except:
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    benchmark_prices = pd.Series(5000 + np.cumsum(np.random.randn(len(dates)) * 10), index=dates)

In [ ]:
# ============================================================
# Cell 6: 因子工程 + 排序标签构造
# ============================================================

def build_features(df):
    feats = pd.DataFrame(index=df.index)
    mom = momentum(df['close'], [5, 10, 20])
    feats = pd.concat([feats, mom], axis=1)
    vol = volatility(df['close'], [10, 20])
    feats = pd.concat([feats, vol], axis=1)
    feats['rsi_14'] = rsi(df['close'], 14)
    macd_df = macd(df['close'])
    feats['macd_hist'] = macd_df['histogram']
    bb = bollinger_bands(df['close'])
    feats['bb_pctb'] = bb['bb_pctb']
    feats['bb_width'] = bb['bb_width']
    ptm = price_to_ma(df['close'], [5, 10, 20])
    feats = pd.concat([feats, ptm], axis=1)
    if 'volume' in df.columns:
        feats['vp_corr'] = volume_price_corr(df['close'], df['volume'])
    if 'high' in df.columns and 'low' in df.columns:
        feats['hl_range'] = high_low_range(df['high'], df['low'], df['close'])
    if 'turnover' in df.columns:
        feats['turnover'] = df['turnover']
    return feats

# 构建因子面板
all_records = []
for sym, sdf in stock_data.items():
    if len(sdf) < 60: continue
    feats = build_features(sdf)
    feats['fwd_return_20d'] = sdf['close'].pct_change(20).shift(-20)
    feats['symbol'] = sym
    all_records.append(feats)

panel = pd.concat(all_records).reset_index()
if 'index' in panel.columns:
    panel.rename(columns={'index': 'date'}, inplace=True)
panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')

FEATURE_COLS = [c for c in panel.columns
                if c not in ['date', 'symbol', 'fwd_return_20d', 'year_month']]

# 构造排序标签: 每月截面内按fwd_return排名分5档
def assign_rank_label(group):
    group = group.copy()
    group['rank_label'] = pd.qcut(group['fwd_return_20d'], q=5, labels=[0,1,2,3,4],
                                   duplicates='drop')
    return group

panel_clean = panel.dropna(subset=FEATURE_COLS + ['fwd_return_20d'])
# 按月分组标注
labeled = panel_clean.groupby('year_month', group_keys=False).apply(assign_rank_label)
labeled = labeled.dropna(subset=['rank_label'])
labeled['rank_label'] = labeled['rank_label'].astype(int)

print(f'因子面板: {labeled.shape[0]} 行, {len(FEATURE_COLS)} 因子')
print(f'排序标签分布:\n{labeled["rank_label"].value_counts().sort_index()}')

In [ ]:
# ============================================================
# Cell 7: LambdaMART 模型训练 (滚动窗口)
# ============================================================

months = sorted(labeled['year_month'].unique())
TRAIN_MONTHS = 12
TOP_K = 5

lgb_rank_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [5, 10],
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'verbose': -1,
    'seed': 42,
}

# 同时训练回归模型作对比
lgb_reg_params = {
    'objective': 'regression',
    'metric': 'mse',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'verbose': -1,
    'seed': 42,
}

rank_selections = {}
reg_selections = {}
all_importances = []

for i in range(TRAIN_MONTHS, len(months) - 1):
    train_period = months[i - TRAIN_MONTHS:i]
    test_month = months[i]

    train_data = labeled[labeled['year_month'].isin(train_period)].copy()
    test_data = labeled[labeled['year_month'] == test_month].copy()

    train_data = train_data.dropna(subset=FEATURE_COLS)
    test_data = test_data.dropna(subset=FEATURE_COLS)

    if len(train_data) < 50 or len(test_data) < 5:
        continue

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_data[FEATURE_COLS].values)
    X_test = scaler.transform(test_data[FEATURE_COLS].values)
    y_train_rank = train_data['rank_label'].values
    y_train_reg = train_data['fwd_return_20d'].values

    # LambdaMART: 需要group信息 (每月为一个query)
    train_groups = train_data.groupby('year_month').size().values

    # 训练排序模型
    dtrain_rank = lgb.Dataset(X_train, label=y_train_rank, group=train_groups)
    rank_model = lgb.train(lgb_rank_params, dtrain_rank, num_boost_round=200)

    # 训练回归模型
    dtrain_reg = lgb.Dataset(X_train, label=y_train_reg)
    reg_model = lgb.train(lgb_reg_params, dtrain_reg, num_boost_round=200)

    # 预测 & 选股
    rank_pred = rank_model.predict(X_test)
    reg_pred = reg_model.predict(X_test)

    test_data = test_data.copy()
    test_data['rank_score'] = rank_pred
    test_data['reg_score'] = reg_pred

    rebal_date = test_data['date'].max()

    # 排序模型选股
    rank_top = test_data.nlargest(TOP_K, 'rank_score')['symbol'].tolist()
    rank_selections[rebal_date] = rank_top

    # 回归模型选股
    reg_top = test_data.nlargest(TOP_K, 'reg_score')['symbol'].tolist()
    reg_selections[rebal_date] = reg_top

    imp = dict(zip(FEATURE_COLS, rank_model.feature_importance(importance_type='gain')))
    all_importances.append(imp)

    if (i - TRAIN_MONTHS) % 6 == 0:
        print(f'  {test_month}: LTR选={rank_top[:3]}..., Reg选={reg_top[:3]}...')

print(f'\nLambdaMART 选股期数: {len(rank_selections)}')
print(f'Regression 选股期数: {len(reg_selections)}')

In [ ]:
# ============================================================
# Cell 8: 回测对比
# ============================================================

all_prices = {sym: sdf['close'] for sym, sdf in stock_data.items()}
backtester = MultiStockBacktester(initial_capital=1_000_000, rebalance_freq='M')

# LambdaMART 回测
rank_result = backtester.run(all_prices, rank_selections, benchmark_prices)
print('=== LambdaMART (排序学习) 回测结果 ===')
for k, v in rank_result['metrics'].items():
    print(f'  {k}: {v}')

# Regression 回测
reg_result = backtester.run(all_prices, reg_selections, benchmark_prices)
print('\n=== LightGBM (回归) 回测结果 ===')
for k, v in reg_result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================

# 1. 策略对比
strategies = {
    'LambdaMART (排序)': rank_result['returns'],
    'LightGBM (回归)': reg_result['returns'],
}
plot_cumulative_comparison(strategies, title='LambdaMART vs Regression - 策略对比')

# 2. LambdaMART 收益曲线
bench_eq = None
if rank_result.get('benchmark_returns') is not None:
    bench_eq = (1 + rank_result['benchmark_returns']).cumprod() * rank_result['equity_curve'].iloc[0]
plot_equity_curve(rank_result['equity_curve'], bench_eq,
                  title='LambdaMART 选股 - 累计收益')

# 3. 回撤
plot_drawdown(rank_result['equity_curve'], title='LambdaMART - 回撤')

# 4. 因子重要性
avg_imp = {}
for col in FEATURE_COLS:
    avg_imp[col] = np.mean([imp.get(col, 0) for imp in all_importances])
plot_factor_importance(avg_imp, title='LambdaMART 因子重要性', top_n=15)

# 5. 绩效对比表
comparison = {}
for k in rank_result['metrics']:
    comparison[k] = f"LTR: {rank_result['metrics'][k]}  |  Reg: {reg_result['metrics'][k]}"
plot_metrics_table(comparison, title='LambdaMART vs Regression 对比')

## 结果讨论

### 策略表现

1. **排序 vs 回归**: LambdaMART 关注股票相对排名而非绝对收益，选股更稳健
2. **NDCG 优化**: 直接优化排名质量指标，更贴合选股任务本质
3. **因子重要性**: 排序模型和回归模型对因子的依赖可能不同

### 局限性

- 20 只股票池较小，排序区分度有限
- 5 档标签构造在小样本下可能不均匀
- 未加入基本面因子 (PE/PB/ROE)

### 改进方向

- 扩大股票池到 CSI300 全样本
- 使用 ListNet/ListMLE 等 listwise 排序方法
- 结合行业中性化和市值中性化
- 加入基本面因子构建更全面的因子体系